# Sketch 1.13 - standalone, portable notebook

Self-contained extract of **sketch_1_13_compare** from the combined *9 sketches*
notebook. The sketch code is unchanged; the shared setup it needs
(imports, data loading) is included below. Chart data is **inlined** (no external
`altair-data` files) so it renders on Linux, macOS and Windows alike.

The searchable name universe is restricted to the ever-top-100 names (plus the
sketch's defaults) so the inlined data stays small; the default view is
unchanged.

## Shared setup

In [ ]:
import altair as alt
import pandas as pd
import numpy as np
import geopandas as gpd  # conda install -c conda-forge geopandas
import json

# Inline the chart data so every HTML output is SELF-CONTAINED and renders on
# all platforms (incl. macOS): external altair-data-*.json URL files are not
# fetched by many notebook frontends, which left the charts blank on Mac. The
# search sketches (1.13 / 2.4 / 2.6) restrict their name universe below so the
# inlined tables stay small.
alt.data_transformers.enable('default')
alt.data_transformers.disable_max_rows()
pass

In [ ]:
# Memory-lean load: another heavy job is running on this machine (~2.5 GB free),
# so we read the CSV with categorical dtypes (15k name strings stored once), derive
# the numeric year from the 122 'annais' categories instead of 3.5M strings, then
# relax categories back to plain objects (the strings stay shared) so every later
# groupby keeps its normal semantics.
raw = pd.read_csv('dpt2020.csv', sep=';',
                  dtype={'sexe': 'int8', 'preusuel': 'category',
                         'annais': 'category', 'dpt': 'category', 'nombre': 'int32'})
cat_years = pd.to_numeric(pd.Index(raw['annais'].cat.categories), errors='coerce').astype('float32')
raw['year'] = np.asarray(cat_years)[raw['annais'].cat.codes]
raw.drop(columns=['annais'], inplace=True)
raw['decade'] = (raw['year'] // 10 * 10)
raw['preusuel'] = raw['preusuel'].astype(object)  # shared strings, plain groupbys
raw['dpt'] = raw['dpt'].astype(object)

records = raw[raw.dpt != 'XX'].copy()
named = records[records.preusuel != '_PRENOMS_RARES'].copy()

dept_total = records.groupby('dpt', as_index=False)['nombre'].sum().rename(
    columns={'nombre': 'dept_total'})

ALL_NAMES = sorted(named.preusuel.unique().tolist())  # full dropdown option list
DECADES = list(range(1900, 2021, 10))
# Shared period options for charts that need an all-years aggregate (2.4).
DECADE_OPTS = [0] + DECADES
DECADE_LABELS = ['All years'] + [str(d) for d in DECADES]
SEX_OPTIONS = ['Mixed', 'Boys', 'Girls']
SEX_LABELS = ['mixed', 'boys', 'girls']

import gc
del raw
gc.collect()

print(f'{len(named):,} named rows; {len(ALL_NAMES):,} distinct names.')
named.sample(3)


## Name-universe restriction *(portability/memory)*

In [ ]:
# Portability + memory: keep the INLINED data small by limiting the name
# universe to names that ever reached the yearly top 100, always unioned with this sketch's own
# default names so the default view is unchanged.
import gc
_EXTRA = [x for x in [] if x in set(named['preusuel'])]
_yr = (named.dropna(subset=['year'])
       .groupby(['year', 'preusuel'], as_index=False)['nombre'].sum())
_yr['rk'] = _yr.groupby('year')['nombre'].rank(ascending=False, method='first')
_universe = set(_yr.loc[_yr['rk'] <= 100, 'preusuel'])
del _yr
POPULAR_NAMES = sorted(_universe.union(_EXTRA))
named = named[named['preusuel'].isin(POPULAR_NAMES)].copy()
ALL_NAMES = POPULAR_NAMES
gc.collect()
print(f'Universe: {len(POPULAR_NAMES):,} names (ever reached the yearly top 100 + defaults); {len(named):,} rows kept.')

## The sketch

### Sketch 1.13 - Compare names: dropdown slots, search, or Top-K of the window

Two ways to choose what is drawn over the grey top-30 context:

- pick names - up to 20 removable dropdown slots spanning all ~15,270 names. All slots default to `(none)`, and the Name slots shown slider controls how many numbered slots are visible and allowed to draw;
- or take the Top-K of the window - the K slider draws the K most popular names *of the current From/To window* as thin coloured curves (K = 0 switches this off).

Two sliders set the year window; the ranking follows it. The sex selector recomputes the Top-K for mixed / boys / girls.


In [ ]:
_ts_base = named.dropna(subset=['year'])
ts_mixte = (_ts_base.groupby(['year', 'preusuel'], as_index=False)['nombre'].sum())
ts_mixte['sex_mode'] = 'Mixed'
ts_sex = (_ts_base.groupby(['year', 'preusuel', 'sexe'], as_index=False)['nombre'].sum())
ts_sex['sex_mode'] = np.where(ts_sex['sexe'].eq(1), 'Boys', 'Girls')
ts_all = pd.concat([
    ts_mixte[['year', 'preusuel', 'nombre', 'sex_mode']],
    ts_sex[['year', 'preusuel', 'nombre', 'sex_mode']],
], ignore_index=True)

top30_by_mode = (ts_all.groupby(['sex_mode', 'preusuel'], as_index=False)['nombre'].sum()
                 .sort_values(['sex_mode', 'nombre'], ascending=[True, False])
                 .groupby('sex_mode').head(30)[['sex_mode', 'preusuel']])
ts_ctx = ts_all.merge(top30_by_mode, on=['sex_mode', 'preusuel'])

NAME_PICK_OPTIONS = ['__NONE__'] + ALL_NAMES
NAME_PICK_LABELS = ['(none)'] + ALL_NAMES
NAME_SLOT_COUNT = 20
NAME_SLOT_COLORS = [
    '#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
    '#b2182b', '#2166ac', '#1b7837', '#b35806', '#542788',
    '#a6611a', '#c51b7d', '#4d4d4d', '#7fbc41', '#35978f',
]
name_slots = alt.param(name='name_slots', value=3,
                       bind=alt.binding_range(min=0, max=NAME_SLOT_COUNT, step=1,
                                              name='Name slots shown  '))
name_param_defs = []
for i in range(1, NAME_SLOT_COUNT + 1):
    param_name = f'name_{i:02d}'
    param_obj = alt.param(
        name=param_name,
        value='__NONE__',
        bind=alt.binding_select(options=NAME_PICK_OPTIONS, labels=NAME_PICK_LABELS,
                                name=f'Name {i:02d}  '),
    )
    name_param_defs.append((i, param_name, param_obj))

name_q = alt.param(name='name_q', value='',
                   bind=alt.binding(input='search', name='Free search  ',
                                    placeholder='e.g. KEVIN'))
sex_113 = alt.param(name='sex_113', value='Mixed',
                    bind=alt.binding_select(options=SEX_OPTIONS, labels=SEX_LABELS,
                                            name='Sex  '))
k_sel = alt.param(name='k_sel', value=8,
                  bind=alt.binding_range(min=0, max=30, step=1,
                                         name='Top-K of the window (0 = off)  '))
w_from = alt.param(name='w_from', value=1900,
                   bind=alt.binding_range(min=1900, max=2020, step=1, name='From year  '))
w_to = alt.param(name='w_to', value=2020,
                 bind=alt.binding_range(min=1900, max=2020, step=1, name='To year  '))
in_window = 'datum.sex_mode == sex_113 && datum.year >= w_from && datum.year <= w_to'

context = alt.Chart(ts_ctx).transform_filter(in_window).mark_line(
    color='#c7c7c7', strokeWidth=0.8, opacity=0.4).encode(
    x=alt.X('year:Q', title='Year', axis=alt.Axis(format='d', tickCount=13)),
    y=alt.Y('nombre:Q', title='Births'), detail='preusuel:N')
wars_df = pd.DataFrame({'year': [1914, 1939], 'lab': ['WWI', 'WWII'],
                        'yy': [ts_ctx['nombre'].max() * 0.98] * 2})
wars = alt.Chart(wars_df).transform_filter('datum.year >= w_from && datum.year <= w_to').mark_rule(
    color='#bbb', strokeDash=[2, 2]).encode(x='year:Q')
wars_lab = alt.Chart(wars_df).transform_filter('datum.year >= w_from && datum.year <= w_to').mark_text(
    color='#777', fontSize=9, dx=3, align='left', baseline='top').encode(
    x='year:Q', y='yy:Q', text='lab:N')

def picked_layer(slot_idx, param_name, colour):
    dy = ((slot_idx - 1) % 5 - 2) * 7
    flt = (f"name_slots >= {slot_idx} && {param_name} != '__NONE__' "
           f"&& datum.preusuel == {param_name} && ({in_window})")
    line = alt.Chart(ts_all).transform_filter(flt).mark_line(strokeWidth=2.5, color=colour).encode(
        x='year:Q', y='nombre:Q',
        tooltip=[alt.Tooltip('preusuel:N', title='Name'),
                 alt.Tooltip('sex_mode:N', title='Sex mode'),
                 alt.Tooltip('year:Q', title='Year', format='d'),
                 alt.Tooltip('nombre:Q', title='Births', format=',')])
    # Label each selected series at its in-window peak; end labels collapse for old names.
    tip = (alt.Chart(ts_all).transform_filter(flt)
           .transform_joinaggregate(maxn='max(nombre)')
           .transform_filter('datum.nombre == datum.maxn')
           .mark_text(align='left', dx=6, dy=dy, fontWeight='bold', color=colour, fontSize=11)
           .encode(x='year:Q', y='nombre:Q', text='preusuel:N'))
    return line + tip

manual_layers = [
    picked_layer(slot_idx, param_name, NAME_SLOT_COLORS[(slot_idx - 1) % len(NAME_SLOT_COLORS)])
    for slot_idx, param_name, _ in name_param_defs
]

topk = (alt.Chart(ts_all).transform_filter(in_window)
        .transform_joinaggregate(tot='sum(nombre)', groupby=['preusuel'])
        .transform_window(rk='dense_rank()', sort=[alt.SortField('tot', order='descending')])
        .transform_filter('datum.rk <= k_sel')
        .mark_line(strokeWidth=1.3, opacity=0.85).encode(
            x='year:Q', y='nombre:Q',
            color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='tableau20')),
            tooltip=[alt.Tooltip('preusuel:N', title='Name'),
                     alt.Tooltip('sex_mode:N', title='Sex mode'),
                     alt.Tooltip('year:Q', title='Year', format='d'),
                     alt.Tooltip('nombre:Q', title='Births', format=',')]))
flt_q = f"name_q != '' && upper(datum.preusuel) == upper(name_q) && ({in_window})"
lay_q = (alt.Chart(ts_all).transform_filter(flt_q).mark_line(strokeWidth=2.8, color='#9467bd')
         .encode(x='year:Q', y='nombre:Q',
                 tooltip=[alt.Tooltip('preusuel:N', title='Name'),
                          alt.Tooltip('sex_mode:N', title='Sex mode'),
                          alt.Tooltip('year:Q', title='Year', format='d'),
                          alt.Tooltip('nombre:Q', title='Births', format=',')]))

sketch_1_13 = alt.layer(context, wars, wars_lab, topk, *manual_layers, lay_q).add_params(
    name_slots, *[param_obj for _, _, param_obj in name_param_defs],
    name_q, sex_113, k_sel, w_from, w_to).properties(
    width=820, height=380,
    title='1.13 - Pick up to 20 names or take the Top-K of the window')

sketch_1_13.save('sketch_1_13_slices.png', ppi=120)
sketch_1_13
